In [0]:
%run ../prod/config

In [0]:
SECRET_SCOPE = get_widget_param("secret_scope", "eventhub")
SECRET_KEY = get_widget_param("secret_key", "eh-connection-string")
EH_NAMESPACE = get_widget_param("eh_namespace", "evhua5816bd")
EH_NAME = get_widget_param("eh_name", "roksolana-wikipedia-recentchange")
CATALOG = get_widget_param("catalog", "dbr_dev_ua5816bd")
SCHEMA_LANDING = get_widget_param("schema_landing", "roksolana_shendiu770")
SCHEMA_BRONZE = get_widget_param("schema_bronze", "roksolana_shendiu770_bronze")
TARGET_TABLE_NAME = get_widget_param("target_table_name", "wikipedia_recentchange_bronze")
CHECKPOINT_SUBDIR = get_widget_param("checkpoint_subdir", "wikipedia_recentchange")
STARTING_OFFSETS = get_widget_param("starting_offsets", "earliest")
MAX_OFFSETS_PER_TRIGGER = get_widget_param("max_offsets_per_trigger", "50000")
FAIL_ON_DATA_LOSS = get_widget_param("fail_on_data_loss", "false")
KAFKA_REQUEST_TIMEOUT_MS = get_widget_param("kafka_request_timeout_ms", "60000")
KAFKA_SESSION_TIMEOUT_MS = get_widget_param("kafka_session_timeout_ms", "30000")

CHECKPOINT_LOCATION = f"/Volumes/{CATALOG}/{SCHEMA_LANDING}/bronze_landing/_checkpoints/{CHECKPOINT_SUBDIR}"
TARGET_TABLE = f"{CATALOG}.{SCHEMA_BRONZE}.{TARGET_TABLE_NAME}"

EH_CONN_STR = get_eventhub_connection_string(SECRET_SCOPE, SECRET_KEY)
KAFKA_OPTIONS = build_kafka_options(
    EH_NAMESPACE, EH_NAME, EH_CONN_STR,
    KAFKA_REQUEST_TIMEOUT_MS, KAFKA_SESSION_TIMEOUT_MS,
    MAX_OFFSETS_PER_TRIGGER, FAIL_ON_DATA_LOSS, STARTING_OFFSETS
)

**Step 1: Idempotency check** – rerun the consumer with `availableNow` and no new
data in Event Hub. Row count before/after should match – checkpoint prevents reprocessing.

In [0]:
%sql
SELECT COUNT(*) FROM dbr_dev_ua5816bd.roksolana_shendiu770_bronze.wikipedia_recentchange_bronze

**result:** 500 -> 500 rows after rerunning with no new data. Checkpoint
correctly prevents reprocessing.

**Step 2: Checkpoint loss risk** – delete checkpoint only, rerun with `startingOffsets=earliest`.
Expect duplicated rows, since Kafka source loses track of already-read offsets.

In [0]:
dbutils.fs.rm(CHECKPOINT_LOCATION, recurse=True)

In [0]:
%sql
SELECT COUNT(*) FROM dbr_dev_ua5816bd.roksolana_shendiu770_bronze.wikipedia_recentchange_bronze

**result:** deleted checkpoint, re-ran with `startingOffsets=earliest`.
Row count stayed at 500 (no duplicate STREAMING UPDATE in history) – Delta's
transaction log provided an additional layer of write idempotency even after
checkpoint loss, preventing duplication.

Note: this differs from the Auto
Loader file-based scenario, where checkpoint loss did cause duplicates – Kafka
offset-based idempotency appears more resilient here.

**Step 3: Safe reload** – cleared checkpoint AND target table together, then reran
the consumer from scratch. This is the correct way to fully reset: clearing checkpoint
alone (Step 2) relied on Delta's write idempotency as a side effect, not a guaranteed
mechanism – clearing both checkpoint and table removes any dependency on that behavior
and guarantees a clean, predictable state.

In [0]:
dbutils.fs.rm(CHECKPOINT_LOCATION, recurse=True)
spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE}")

In [0]:
%sql
SELECT COUNT(*) FROM dbr_dev_ua5816bd.roksolana_shendiu770_bronze.wikipedia_recentchange_bronze

**result:** 500 rows – clean, predictable reload confirmed